# SparseVILA Demo on LLaVA-1.5-7B

Reference implementation of [SparseVILA (ICCV 2025)](https://arxiv.org/abs/2510.17777) applied to LLaVA-1.5-7B. This notebook demonstrates end-to-end inference with and without the two-stage decoupled sparsity (Algorithm 2 + Algorithm 3 from the paper).

For design, see `docs/superpowers/specs/2026-05-15-sparsevila-impl-design.md` in the repo.

In [ ]:
# Replace <repo-url> with the actual GitHub URL before running.
!git clone <repo-url>
%cd "[man] SparseVILA"
!git submodule update --init --recursive
!pip install -q -e ./flash-colreduce
!pip install -q -e ./third_party/LLaVA --no-deps
!pip install -q -e .

In [ ]:
import torch
from PIL import Image
from sparsevila import load_sparse_llava, SparseVILAConfig

In [ ]:
cfg_vanilla = SparseVILAConfig()  # zero ratios
model_v, processor_v = load_sparse_llava(
    "liuhaotian/llava-v1.5-7b", config=cfg_vanilla,
    dtype=torch.float16, device="cuda")

In [ ]:
cfg_sparse = SparseVILAConfig(
    encoder_prune_ratio=0.5,
    decode_retrieval_ratio=0.75,
)
model_s, processor_s = load_sparse_llava(
    "liuhaotian/llava-v1.5-7b", config=cfg_sparse,
    dtype=torch.float16, device="cuda")

In [ ]:
from io import BytesIO
import requests
img = Image.open(BytesIO(requests.get(
    "https://llava-vl.github.io/static/images/view.jpg").content)).convert("RGB")
prompt = "What is unusual about this image?"

In [ ]:
torch.manual_seed(0)
inputs_v = processor_v(images=img, text=prompt, return_tensors="pt").to("cuda")
out_v = model_v.generate(**inputs_v, max_new_tokens=80, do_sample=False)
text_v = processor_v.decode(out_v[0], skip_special_tokens=True)
print("VANILLA:
", text_v)

In [ ]:
torch.manual_seed(0)
inputs_s = processor_s(images=img, text=prompt, return_tensors="pt").to("cuda")
out_s = model_s.generate(**inputs_s, max_new_tokens=80, do_sample=False)
text_s = processor_s.decode(out_s[0], skip_special_tokens=True)
print("SPARSEVILA:
", text_s)

In [ ]:
# Re-load with zero config, compare output IDs to a second zero-config load.
# If both produce identical output IDs for 50 tokens, the wrapping is byte-equivalent
# to vanilla LLaVA — proves no regression.
m1, p1 = load_sparse_llava("liuhaotian/llava-v1.5-7b",
                           config=SparseVILAConfig(), dtype=torch.float16, device="cuda")
m2, p2 = load_sparse_llava("liuhaotian/llava-v1.5-7b",
                           config=SparseVILAConfig(), dtype=torch.float16, device="cuda")
inp = p1(images=img, text=prompt, return_tensors="pt").to("cuda")
torch.manual_seed(0); o1 = m1.generate(**inp, max_new_tokens=50, do_sample=False)
torch.manual_seed(0); o2 = m2.generate(**inp, max_new_tokens=50, do_sample=False)
assert torch.equal(o1, o2), "Two zero-config loads must produce identical output"
print("Golden check OK")

## Next steps

Not included in this reference impl:
- Multi-turn conversation evaluation (Algorithm 4)
- Accuracy benchmarks (POPE, GQA, etc.)
- Latency benchmarks vs vanilla LLaVA
- AWQ / SmoothQuant quantization
- LLaVA-NeXT anyres tiling

See `docs/superpowers/specs/2026-05-15-sparsevila-impl-design.md` for the deferred-feature roadmap.